In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Library

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()
import gzip
import json

# 데이터 시각화 (EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# 텍스트 데이터 전처리 (nltk)
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
# nltk download
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# Utils

In [ ]:
# 데이터 불러오기
def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield json.loads(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

In [ ]:
# Top 5 core Filtering
def top_n_filtering(data, chk=5):
    cnt = 1
    while True:
        # 사용자 필터링
        user_counts = data['user_id'].value_counts()
        users_to_remove = user_counts[user_counts < chk].index
        data = data[~data['user_id'].isin(users_to_remove)]

        # 아이템 필터링
        item_counts = data['asin'].value_counts()
        items_to_remove = item_counts[item_counts < chk].index
        data = data[~data['asin'].isin(items_to_remove)]

        print(f"Epoch {cnt}")
        print(f" - Remaining users: {data['user_id'].nunique()}")
        print(f" - Remaining items: {data['asin'].nunique()}")
        print(f" - Remaining reviews: {len(data)}")

        # 두 조건 모두 만족 시 종료
        if (len(users_to_remove) == 0) and (len(items_to_remove) == 0):
            break

        cnt += 1

    return data

In [ ]:
# 사용자별 Top-N 필터링
def user_top_n_filtering(data, chk=5):
    cnt = 1
    while True:
        # 사용자별 리뷰 수 카운트
        user_counts = data['user_id'].value_counts()
        users_to_remove = user_counts[user_counts < chk].index
        if len(users_to_remove) == 0:
            break

        data = data[~data['user_id'].isin(users_to_remove)]

        print(f"Epoch {cnt}")
        print(f" - Remaining users: {data['user_id'].nunique()}")
        print(f" - Remaining items: {data['asin'].nunique()}")
        print(f" - Remaining reviews: {len(data)}")

        cnt += 1
    return data

# Load Dataset

In [ ]:
# 리뷰 데이터셋 확인
df = getDF('/content/drive/MyDrive/Dataset/Amazon_Dataset/Review/Cell_Phones_and_Accessories.jsonl.gz')

print(df.shape)
df.head()

In [ ]:
# 아이템 메타 데이터셋 확인
meta_df = getDF('/content/drive/MyDrive/Dataset/Amazon_Dataset/Meta/meta_Cell_Phones_and_Accessories.jsonl.gz')

print(meta_df.shape)
meta_df.head()

In [ ]:
# 공통으로 존재하는 parent_asin Column으로 결합.
merged = df.merge(meta_df, on='parent_asin', how='left')

print(merged.shape)
merged.head()

In [ ]:
# 사용자가 남긴 img가 있는 데이터만 분리
merged_img = merged.loc[merged['images_x'].apply(lambda x: len(x) > 0)]
merged_img = merged_img.reset_index(drop=True)

print(merged_img.shape)
merged_img.head()

In [ ]:
# 리뷰 데이터셋 확인
merged_img.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_and_Accessories_Img_Filtered.pkl')

In [ ]:
merged_img.to_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_and_Accessories_Img_Filtered.json')

# EDA

In [ ]:
# 데이터 수집 기간 확인
date_min = pd.to_datetime(merged_img['timestamp'],unit='ms').min()
date_max = pd.to_datetime(merged_img['timestamp'],unit='ms').max()
date_min,date_max

In [ ]:
# year, month column 생성
merged_img.loc[:, 'year'] = pd.to_datetime(merged_img['timestamp'], unit='ms').dt.year
merged_img.loc[:, 'month'] = pd.to_datetime(merged_img['timestamp'], unit='ms').dt.month

In [ ]:
# 연도별로 축적된 데이터가 얼마나 있는지 확인
merged_img.groupby('year').agg({'rating':'count'}).reset_index().plot(x='year',kind='bar')

# Top N Filtering
- review와 image 데이터가 모두 존재하는 데이터를 우선 필터링 하였으므로 Dataset의 Sparsity가 있는 상황임.
- user top 5 filtering을 적용하거나, Top-N filtering을 적용하지 않음.

In [ ]:
# 전체 img_data
merged_img.shape

In [ ]:
# review를 중심으로 중복되는 데이터 삭제
merged_img = merged_img.drop_duplicates(['text'], keep='first')

print(merged_img.shape)
merged_img.head()

In [ ]:
top3 = top_n_filtering(merged_img, 3)

# index 초기화
top3 = top3.reset_index(drop=True)
top3.shape

# Top 3 Image EDA

In [ ]:
# 연도별로 축적된 데이터가 얼마나 있는지 확인
top3.groupby('year').agg({'rating':'count'}).reset_index().plot(x='year',kind='bar')

## 2014 image

In [ ]:
for data in top3.loc[top3['year']==2014, 'images_x'].values[:50]:
    print(data[0]['small_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2014, 'images_x'].values[:50]:
    print(data[0]['medium_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2014, 'images_x'].values[:50]:
    print(data[0]['large_image_url'])

## 2015 image

In [ ]:
for data in top3.loc[top3['year']==2015, 'images_x'].values[:50]:
    print(data[0]['small_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2015, 'images_x'].values[:50]:
    print(data[0]['medium_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2015, 'images_x'].values[:50]:
    print(data[0]['large_image_url'])

## 2016 image

In [ ]:
for data in top3.loc[top3['year']==2016, 'images_x'].values[:50]:
    print(data[0]['small_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2016, 'images_x'].values[:50]:
    print(data[0]['medium_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2016, 'images_x'].values[:50]:
    print(data[0]['large_image_url'])

## 2017 image

In [ ]:
for data in top3.loc[top3['year']==2017, 'images_x'].values[:50]:
    print(data[0]['small_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2017, 'images_x'].values[:50]:
    print(data[0]['medium_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2017, 'images_x'].values[:50]:
    print(data[0]['large_image_url'])

## 2018 image

In [ ]:
for data in top3.loc[top3['year']==2018, 'images_x'].values[:50]:
    print(data[0]['small_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2018, 'images_x'].values[:50]:
    print(data[0]['medium_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2018, 'images_x'].values[:50]:
    print(data[0]['large_image_url'])

## 2019 image

In [ ]:
for data in top3.loc[top3['year']==2019, 'images_x'].values[:50]:
    print(data[0]['small_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2019, 'images_x'].values[:50]:
    print(data[0]['medium_image_url'])

In [ ]:
for data in top3.loc[top3['year']==2019, 'images_x'].values[:50]:
    print(data[0]['large_image_url'])

# Save Result

In [ ]:
top3.shape

In [ ]:
top3.to_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_and_Accessories_Img_Filtered_Top3.json')